# Czas koncentracji zlewni

Czas koncentracji (tc) to czas potrzebny na przepływ wody z hydraulicznie najodleglejszego punktu zlewni do przekroju zamykającego (USDA-NRCS 2010, NEH 630 rozdz. 15, s. 15-3).

Hydrolog oferuje sześć metod obliczania tc: Kirpich, NRCS, Giandotti, FAA, Kerby i Kerby–Kirpich. Ten notatnik pokazuje trzy z nich:
1. **Kirpich** - bardzo małe zlewnie rolnicze (do 0,45 km²)
2. **NRCS** - równanie opóźnienia z CN (pokrycie terenu i gleby), zlewnie do 24 km²
3. **Giandotti** - zlewnie środkowych i północnych Włoch o powierzchni 170–70 000 km²

FAA i Kerby dają czas spływu powierzchniowego (stokowego), a Kerby–Kirpich sumę spływu powierzchniowego i przepływu w korycie; wszystkie sześć liczy z parametrów zlewni `WatershedParameters.calculate_tc()` i polecenie `hydrolog tc watershed`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

from hydrolog.time import ConcentrationTime

## Przykładowa zlewnia

Parametry zlewni do obliczeń. Każda metoda bierze dane, które definiuje jej źródło:
- Kirpich: długość najdłuższej drogi spływu i jej spadek (różnica wysokości jej początku i przekroju zamykającego przez długość),
- NRCS: tę samą długość, średni spadek terenu zlewni Y (nie spadek drogi spływu) i CN,
- Giandotti: powierzchnię, długość cieku głównego i różnicę średniej wysokości zlewni i wysokości przekroju zamykającego.

In [ ]:
# Parametry zlewni
area_km2 = 45.0                # Powierzchnia [km²]
flow_path_length_km = 12.0     # Długość najdłuższej drogi spływu [km] (Kirpich, NRCS)
head_elevation_m = 530.0       # Wysokość początku najdłuższej drogi spływu [m n.p.m.]
outlet_elevation_m = 180.0     # Wysokość przekroju zamykającego [m n.p.m.]
mean_elevation_m = 340.0       # Średnia wysokość zlewni [m n.p.m.] (Giandotti)
channel_length_km = 11.0       # Długość cieku głównego [km] (Giandotti)
mean_slope_m_per_m = 0.06      # Średni spadek terenu zlewni [m/m] (NRCS: Y)
cn = 72                        # Curve Number

# Spadek najdłuższej drogi spływu (Kirpich): różnica wysokości przez długość
flow_path_slope_m_per_m = (head_elevation_m - outlet_elevation_m) / (flow_path_length_km * 1000)
# Wysokość średnia nad przekrojem zamykającym (Giandotti: H_m - H_0)
mean_height_above_outlet_m = mean_elevation_m - outlet_elevation_m

print("PARAMETRY ZLEWNI")
print("=" * 50)
print(f"Powierzchnia: {area_km2} km²")
print(f"Najdłuższa droga spływu: {flow_path_length_km} km, spadek {flow_path_slope_m_per_m:.4f} m/m")
print(f"Ciek główny: {channel_length_km} km")
print(f"Średni spadek terenu Y: {mean_slope_m_per_m:.3f} m/m ({mean_slope_m_per_m * 100:.1f}%)")
print(f"Średnia wysokość nad przekrojem zamykającym: {mean_height_above_outlet_m:.0f} m")
print(f"CN: {cn}")

## 1. Wzór Kirpicha

Wzór dla bardzo małych zlewni rolniczych w Tennessee (Kirpich 1940; Fang i in. 2005, wzór 3.14):

$$t_c = 0.0078 \cdot L^{0.77} \cdot S^{-0.385}$$

Gdzie:
- t_c - czas koncentracji [min]
- L - długość najdłuższej drogi spływu [ft] (1 ft = 0,3048 m)
- S - jej spadek [m/m]

**Uwaga:** W Hydrolog podajemy L w km, a funkcja przelicza ją na stopy dokładnie. Długość 12 km leży poza danymi wzoru (350–4000 ft, czyli 0,107–1,22 km), więc Hydrolog ostrzega (`OutsideTypicalRange`).

In [ ]:
# Wzór Kirpicha
tc_kirpich = ConcentrationTime.kirpich(
    length_km=flow_path_length_km,
    slope_m_per_m=flow_path_slope_m_per_m,
)

print(f"Czas koncentracji (Kirpich): {tc_kirpich:.1f} min ({tc_kirpich/60:.2f} h)")

## 2. Wzór NRCS

Równanie opóźnienia NRCS (Mockus 1961) uwzględnia Curve Number, co pozwala odwzorować wpływ użytkowania terenu i gleb. Czas koncentracji (USDA-NRCS 2010, NEH 630 rozdz. 15, wzór 15-4b):

$$t_c = \frac{\ell^{0.8} \cdot (S+1)^{0.7}}{1140 \cdot Y^{0.5}}$$

Gdzie:
- t_c - czas koncentracji [h]
- ℓ - długość najdłuższej drogi spływu [ft]
- S = 1000/CN - 10 - retencja [in]
- Y - średni spadek terenu zlewni [%], „nie mylić ze spadkiem drogi spływu” (NEH 630 rozdz. 15, s. 15-5)

Hydrolog przyjmuje długość w km i spadek w m/m i przelicza je dokładnie (t_c = L/0,6, wzór 15-3, gdzie L to opóźnienie z wzoru 15-4a).

In [ ]:
# Wzór NRCS: długość najdłuższej drogi spływu i średni spadek terenu zlewni
tc_nrcs = ConcentrationTime.nrcs(
    length_km=flow_path_length_km,
    slope_m_per_m=mean_slope_m_per_m,
    cn=cn,
)

print(f"Czas koncentracji (NRCS): {tc_nrcs:.1f} min ({tc_nrcs/60:.2f} h)")

## 3. Wzór Giandottiego

Metoda opracowana dla zlewni środkowych i północnych Włoch (170–70 000 km²; Giandotti 1934, za Grimaldi i in. 2012). Uwzględnia powierzchnię zlewni i jej średnią wysokość nad przekrojem zamykającym (Giandotti 1934, za Autorità di bacino del fiume Po 2001, §6.1):

$$t_c = \frac{4\sqrt{A} + 1.5L}{0.8\sqrt{H_m - H_0}}$$

Gdzie:
- t_c - czas koncentracji [h]
- A - powierzchnia zlewni [km²]
- L - długość cieku głównego [km]
- H_m - średnia wysokość zlewni [m n.p.m.]
- H_0 - wysokość przekroju zamykającego [m n.p.m.]

H_m - H_0 to nie deniwelacja zlewni (różnica wysokości najwyższego i najniższego punktu).

In [ ]:
# Wzór Giandottiego: H_m - H_0
tc_giandotti = ConcentrationTime.giandotti(
    area_km2=area_km2,
    length_km=channel_length_km,
    elevation_diff_m=mean_height_above_outlet_m,
)

print(f"Czas koncentracji (Giandotti): {tc_giandotti:.1f} min ({tc_giandotti/60:.2f} h)")

## 4. Porównanie metod

In [ ]:
print("PORÓWNANIE METOD")
print("=" * 50)
print(f"{'Metoda':<20} {'tc [min]':>12} {'tc [h]':>12}")
print("-" * 50)
print(f"{'Kirpich':<20} {tc_kirpich:>12.1f} {tc_kirpich/60:>12.2f}")
print(f"{'NRCS':<20} {tc_nrcs:>12.1f} {tc_nrcs/60:>12.2f}")
print(f"{'Giandotti':<20} {tc_giandotti:>12.1f} {tc_giandotti/60:>12.2f}")
print("-" * 50)

# Średnia i rozstęp
tc_values = [tc_kirpich, tc_nrcs, tc_giandotti]
tc_mean = sum(tc_values) / len(tc_values)
tc_range = max(tc_values) - min(tc_values)

print(f"{'Średnia':<20} {tc_mean:>12.1f} {tc_mean/60:>12.2f}")
print(f"Rozstęp: {tc_range:.1f} min ({tc_range/tc_mean*100:.1f}% średniej)")

## 5. Wpływ parametrów na tc

In [ ]:
# Wpływ długości drogi spływu (Kirpich), w zakresie danych wzoru
print("Wpływ długości drogi spływu (Kirpich, S=0.03):")
print("-" * 40)
for L in [0.2, 0.4, 0.6, 0.8, 1.2]:
    tc = ConcentrationTime.kirpich(length_km=L, slope_m_per_m=0.03)
    print(f"L = {L:.1f} km  →  tc = {tc:6.1f} min")

In [ ]:
# Wpływ spadku (Kirpich), w zakresie danych wzoru
print("\nWpływ spadku (Kirpich, L=1 km):")
print("-" * 40)
for S in [0.025, 0.03, 0.05, 0.075, 0.10]:
    tc = ConcentrationTime.kirpich(length_km=1, slope_m_per_m=S)
    print(f"S = {S:.3f} m/m ({S*100:.1f}%)  →  tc = {tc:6.1f} min")

In [ ]:
# Wpływ CN (NRCS), CN 50–95 (Mockus 1961)
print("\nWpływ CN (NRCS, L=10 km, Y=0.03):")
print("-" * 40)
for CN in [50, 60, 70, 80, 90, 95]:
    tc = ConcentrationTime.nrcs(length_km=10, slope_m_per_m=0.03, cn=CN)
    print(f"CN = {CN}  →  tc = {tc:6.1f} min")

## 6. Wizualizacja

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Wpływ długości (zakres danych Kirpicha: 0,107–1,22 km)
    lengths = np.linspace(0.11, 1.2, 50)
    tc_vals = [ConcentrationTime.kirpich(float(L), 0.03) for L in lengths]
    axes[0].plot(lengths, tc_vals, 'b-', linewidth=2)
    axes[0].set_xlabel('Długość drogi spływu [km]')
    axes[0].set_ylabel('Czas koncentracji [min]')
    axes[0].set_title('Wpływ długości (Kirpich, S=3%)')
    axes[0].grid(True, alpha=0.3)
    
    # Wpływ spadku (zakres danych Kirpicha: 0,025–0,1)
    slopes = np.linspace(0.025, 0.1, 50)
    tc_vals = [ConcentrationTime.kirpich(1.0, float(S)) for S in slopes]
    axes[1].plot(slopes * 100, tc_vals, 'g-', linewidth=2)
    axes[1].set_xlabel('Spadek [%]')
    axes[1].set_ylabel('Czas koncentracji [min]')
    axes[1].set_title('Wpływ spadku (Kirpich, L=1 km)')
    axes[1].grid(True, alpha=0.3)
    
    # Wpływ CN (CN 50–95, Mockus 1961)
    cns = np.linspace(50, 95, 50)
    tc_vals = [ConcentrationTime.nrcs(length_km=10, slope_m_per_m=0.03, cn=float(c)) for c in cns]
    axes[2].plot(cns, tc_vals, 'r-', linewidth=2)
    axes[2].set_xlabel('Curve Number (CN)')
    axes[2].set_ylabel('Czas koncentracji [min]')
    axes[2].set_title('Wpływ CN (NRCS, L=10 km, Y=3%)')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("Matplotlib nie jest zainstalowany.")

## Podsumowanie

| Metoda | Parametry wejściowe | Zakres danych, z których pochodzi wzór |
|--------|--------------------|--------------|
| Kirpich | L i S najdłuższej drogi spływu | Bardzo małe zlewnie rolnicze w Tennessee: 0,005–0,45 km² (1,25–112 akrów; Kirpich 1940, za USDA-NRCS 2010, NEH 630 rozdz. 15, i Roussel i in. 2005), długość drogi spływu 350–4000 ft (0,107–1,22 km) i spadek 0,025–0,1 m/m (Mota i Kobiyama 2015, s. 58); sam Kirpich podawał zastosowanie do 200 akrów (0,81 km²; Fang i in. 2005, Mota i Kobiyama 2015) |
| NRCS | L najdłuższej drogi spływu, średni spadek terenu Y, CN | Zlewnie do 24 km² (od 1,3 akra do 9,2 mi²), CN 50–95 (Mockus 1961, za USDA-NRCS 2010, NEH 630 rozdz. 15) |
| Giandotti | A, L cieku głównego, H_m - H_0 | Zlewnie środkowych i północnych Włoch: 170–70 000 km² (Giandotti 1934, za Grimaldi i in. 2012) |

O wartości spoza tych zakresów Hydrolog ostrzega (`outside_typical_range`); powierzchnię sprawdza, gdy ją zna (`tc watershed`, `hydrograph generate`). Zakresy wszystkich sześciu metod podaje `hydrolog commands schema --command "tc watershed"` (pole `typical_ranges`), a ich wzory i stałe `docs/FORMULAS.md`. Przykładowa zlewnia tego notatnika (45 km², najdłuższa droga spływu 12 km) leży poza zakresem każdego z trzech wzorów: jej wyniki pokazują tylko działanie funkcji.

**Zalecenia:**
- Dla bardzo małych zlewni rolniczych (do 0,45 km²): **Kirpich**
- Gdy znany jest CN, dla zlewni do 24 km²: **NRCS**
- Dla dużych zlewni (od 170 km²): **Giandotti**
- W razie wątpliwości: użyj średniej z kilku metod